# Nearest Neighbor Classifier on CIFAR-10

This is the classic baseline classifier from Stanford's CS231n course.

It does **not** use any deep learning — it just:
1. **Memorizes** every training image (`train`)
2. For every test image, finds the **single most similar** training image by raw pixel distance, and copies its label (`predict`)

**Why build this at all?** It's a rock-bottom baseline that reveals two core problems every later algorithm (KNN → Linear Classifiers → CNNs) is designed to fix:
- Prediction is painfully slow (has to scan the whole training set every time)
- Raw pixel distance is a poor measure of "similarity" between images


## 1. Imports

In [ ]:
import numpy as np
import pickle
import os
import urllib.request
import tarfile
import matplotlib.pyplot as plt


## 2. Download and load CIFAR-10

This downloads the official CIFAR-10 python batches (~170MB) the first time you run it.

In [ ]:
def download_cifar10(data_dir="./cifar10_data"):
    url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    tar_path = os.path.join(data_dir, "cifar-10-python.tar.gz")
    extracted_dir = os.path.join(data_dir, "cifar-10-batches-py")

    if os.path.exists(extracted_dir):
        return extracted_dir

    os.makedirs(data_dir, exist_ok=True)
    print("Downloading CIFAR-10 (~170MB)...")
    urllib.request.urlretrieve(url, tar_path)

    print("Extracting...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(data_dir)

    return extracted_dir


def load_cifar_batch(filename):
    with open(filename, "rb") as f:
        batch = pickle.load(f, encoding="bytes")
        images = batch[b"data"]                     # shape: (10000, 3072)
        labels = batch[b"labels"]
        images = images.reshape(10000, 3, 32, 32).transpose(0, 2, 3, 1)  # NHWC
        return images, np.array(labels)


def load_cifar10(data_dir="./cifar10_data"):
    folder = download_cifar10(data_dir)

    xs, ys = [], []
    for i in range(1, 6):
        x, y = load_cifar_batch(os.path.join(folder, f"data_batch_{i}"))
        xs.append(x)
        ys.append(y)
    X_train = np.concatenate(xs)
    y_train = np.concatenate(ys)

    X_test, y_test = load_cifar_batch(os.path.join(folder, "test_batch"))

    return X_train, y_train, X_test, y_test


In [ ]:
X_train, y_train, X_test, y_test = load_cifar10()
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


## 3. Peek at a few images

In [ ]:
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i])
    ax.set_title(class_names[y_train[i]])
    ax.axis("off")
plt.show()


## 4. The Nearest Neighbor Classifier

- `train()` just stores the data → **O(1)**, no real computation ("Machine learning!")
- `predict()` compares each test image against **every** training image → **O(N)**, the expensive part


In [ ]:
class NearestNeighbor:
    def __init__(self):
        self.Xtr = None
        self.ytr = None

    def train(self, images, labels):
        """Memorize all data and labels."""
        self.Xtr = images
        self.ytr = labels

    def predict(self, test_images, distance="L1"):
        """Predict the label of the most similar training image."""
        num_test = test_images.shape[0]
        y_pred = np.zeros(num_test, dtype=self.ytr.dtype)

        Xtr_flat = self.Xtr.reshape(self.Xtr.shape[0], -1).astype(np.float64)

        for i in range(num_test):
            test_vec = test_images[i].reshape(-1).astype(np.float64)

            if distance == "L1":
                distances = np.sum(np.abs(Xtr_flat - test_vec), axis=1)
            elif distance == "L2":
                distances = np.sqrt(np.sum((Xtr_flat - test_vec) ** 2, axis=1))
            else:
                raise ValueError("distance must be 'L1' or 'L2'")

            nearest_index = np.argmin(distances)
            y_pred[i] = self.ytr[nearest_index]

            if (i + 1) % 50 == 0:
                print(f"  predicted {i + 1}/{num_test} test images...")

        return y_pred


## 5. Run it end-to-end

Using a small subset (5000 train / 200 test) to keep runtime reasonable in pure numpy —
this slowness is itself the point being demonstrated.


In [ ]:
N_TRAIN = 5000
N_TEST = 200

X_train_sub, y_train_sub = X_train[:N_TRAIN], y_train[:N_TRAIN]
X_test_sub, y_test_sub = X_test[:N_TEST], y_test[:N_TEST]

nn = NearestNeighbor()

print("Training (memorizing data)...")
nn.train(X_train_sub, y_train_sub)   # instant

print("Predicting (comparing every test image against all training images)...")
y_pred = nn.predict(X_test_sub, distance="L1")   # slow


In [ ]:
accuracy = np.mean(y_pred == y_test_sub)
print(f"Accuracy: {accuracy * 100:.2f}%")


## 6. Visually inspect some predictions

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_test_sub[i])
    correct = y_pred[i] == y_test_sub[i]
    color = "green" if correct else "red"
    ax.set_title(f"pred: {class_names[y_pred[i]]}\ntrue: {class_names[y_test_sub[i]]}",
                 color=color, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 7. Why this matters — the motive behind it

1. **It's a floor, not a ceiling.** Before building a CNN, you need a dumb baseline to know if your "smart" model is actually learning anything.

2. **It exposes the train/predict tradeoff problem.** `train()` is instant, `predict()` is painfully slow — it scans the *entire* training set for every single prediction. Real-world systems want the opposite: slow, offline training and fast, real-time inference. This motivates shifting the heavy computation into training (as linear classifiers and CNNs do).

3. **It proves raw pixel distance is a bad similarity measure.** Two images of the same cat — one shifted a few pixels, or slightly brighter — can have a huge L1/L2 distance, while an unrelated image with a similar-colored background might look "closer." This motivates learning actual *features* (edges, textures, shapes) instead of comparing raw pixels — exactly what convolutional filters do.

4. **It's the simplest possible classifier**, making it the cleanest way to introduce core ML vocabulary (train set, test set, distance metric, accuracy) before adding complexity like gradients, loss functions, or backpropagation.

So the "motive" isn't that anyone deploys Nearest Neighbor in production — it's a teaching tool that sets a baseline and reveals exactly the two problems (speed, and meaningless pixel comparison) that every later algorithm in the course is built to solve.
